In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import re

In [7]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [8]:
data_list = []
url = "https://www.parliament.bg/en/MP"
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") 
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")
# options.headless = True

In [9]:
def get_data(slug_name):
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    time.sleep(2)
    driver.find_element(By.XPATH, f'/html/body/div/nav/div/ul/li[2]/a').click()
    time.sleep(2)
    list1 = driver.find_elements(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div[1]/p/div[2]/div/div[2]/a')
    for i in range(1, len(list1)+1):
        careerInfo = ""
        additionalInfo = ""
        placeOfBirthCity =""
        placeOfBirthCountry=""
        languagesKnown =""
        telephoneNos = ""
        data_dict = {}
        listOfCareerInfo = []
        fullName = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div[1]/p/div[2]/div[{i}]/div[2]/a').text.title()
        print(fullName)
        careerInfoDesignation = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div[1]/p/div[2]/div[{i}]/div[2]/div[1]').text
        print(careerInfoDesignation)
        link = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div[1]/p/div[2]/div[{i}]/div[2]/a').get_attribute("href")
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[1])
        driver.get(link)
        time.sleep(2)
        image = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[1]/img').get_attribute("src")
        print(image)
        list2 = driver.find_elements(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li')
        for j in range(1, len(list2)+1):
            headings = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
            if "Date of birth " in headings:
                dob = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                dob = dob.split(":")[1].strip()
                if "," in dob:
                    placeOfBirthCity = dob.split(",")[1].strip()
                    try:
                        placeOfBirthCountry = dob.split(",")[2].strip()
                    except:
                        pass
                    dob = dob.split(",")[0].strip()
                print(dob)
            elif "Profession: " in headings:
                profession = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                profession = profession.split(":")[1].strip()
                if profession != "":
                    additionalInfo = "Profession: " + profession + "; " + additionalInfo
            elif "Languages: " in headings:
                languagesKnown = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                languagesKnown = languagesKnown.split(":")[1].strip()
                print(languagesKnown)
            elif "Political force: " in headings:
                politicalForce = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                politicalForce = politicalForce.split(":")[1].strip()
                politicalParty = politicalForce.split(",")[0].strip()
                if politicalForce != "":
                    additionalInfo = "Political Force: "+  politicalForce + "; " + additionalInfo
            elif "E-mail: " in headings:
                emails = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                emails = emails.split(":")[1].strip()
            elif "Phone:: " in headings:
                telephoneNos = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                telephoneNos = telephoneNos.split("::")[1].strip()
            elif "Member of the previos NA:" in headings:
                members = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[1]/div/div[2]/ul/li[{j}]').text
                members = members.split(":")[1].strip()
                members = members.replace(";", ",")
                if members != "":
                    additionalInfo = "Member of the previos NA: " + members + "; " + additionalInfo

        print(additionalInfo)
#         list3 = driver.find_elements(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div/p/ul/li')
#         for k in range(1, len(list3)+1):
#             careerInfo = careerInfo + "; " +  driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div/p/ul/li[{k}]').text
#         careerInfo = careerInfo.split("\n")
        info = []
        careerInfo = driver.find_element(By.XPATH, f'/html/body/div/main/div/div/div[2]/div[2]/div/div[2]/div/p/ul').text.split("\n")
        print(range(0, len(careerInfo)-2))
        for k in range(0, len(careerInfo)-3, 2):
            info.append(careerInfo[k]+"; " + careerInfo[k+1])
#             print(info)
        for l in range(0, len(info)):
            temp_dict = {}
            startDate = ""
            endDate = ""
            terms = info[l].split(";")[1].strip()
            roles = info[l].split(";")[0].strip()
            startDate = terms.split("-")[0].strip()
            endDate = terms.split("-")[1].strip()
            if roles:
                temp_dict['roles'] = roles
            if terms:
                temp_dict['terms'] = terms
            if startDate:
                temp_dict['startDate'] = startDate
            if endDate:
                temp_dict['endDate'] = endDate
            listOfCareerInfo.append(temp_dict)
        summary = fullName + " is the " + careerInfoDesignation + " of the German Parliament " + "and belongs to the political party " + politicalParty
        print("*"*50)
        driver.close()
        driver.switch_to.window(driver.window_handles[0])
        if fullName:
            data_dict['fullName'] = fullName
        if careerInfoDesignation:
            data_dict['careerInfoDesignation'] = careerInfoDesignation
        if politicalParty:
            data_dict['politicalParty'] = politicalParty
        if image:
            data_dict['image'] = image
        if dob:
            data_dict['dob'] = dob
        if placeOfBirthCity:
            data_dict['placeOfBirthCity'] = placeOfBirthCity
        if placeOfBirthCountry:
            data_dict['placeOfBirthCountry'] = placeOfBirthCountry
        if languagesKnown:
            data_dict['languagesKnown'] = languagesKnown
        if emails:
            data_dict['emails'] = emails
        if telephoneNos:
            data_dict['telephoneNos'] = telephoneNos
        if listOfCareerInfo:
            data_dict['listOfCareerInfo'] = listOfCareerInfo
        if additionalInfo:
            data_dict['additionalInfo'] = additionalInfo
        if summary:
            data_dict['summary'] = summary
        data_list.append(data_dict)
    return data_list


In [10]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

Adlen Shukri Shevked
Member
https://www.parliament.bg/images/Assembly/3589.png
24/11/1969
English; Russian; Turkish;
Member of the previos NA: 44th NA, 45th NA, 46th NA,; Political Force: Movement for Rights and Freedoms - MRF, 14.00 %; Profession: Economist;; 
range(0, 78)
**************************************************
Ahmed Redzhebov Ahmedov
Member
https://www.parliament.bg/images/Assembly/3637.png
17/07/1974
Russian; Turkish;
Member of the previos NA: 43rd NA, 44th NA, 45th NA, 46th NA,; Political Force: Movement for Rights and Freedoms - MRF, 14.00 %; Profession: Engineer;; 
range(0, 44)
**************************************************
Aleksandar Koychev Ivanov
Member
https://www.parliament.bg/images/Assembly/3654.png
29/08/1982
English; German;
Member of the previos NA: 44th NA, 46th NA,; Political Force: GERB-UDF, 25.00 %; Profession: Economist;; 
range(0, 34)
**************************************************
Aleksandar Nestorov Nestorov
Member
https://www.parliament.bg/im

https://www.parliament.bg/images/Assembly/3590.png
5/08/1986
English; German; Turkish;
Member of the previos NA: 45th NA, 46th NA,; Political Force: Movement for Rights and Freedoms - MRF, 14.00 %; Profession: Economist;; 
range(0, 46)
**************************************************
Blagovest Chanev Belev
Member
https://www.parliament.bg/images/Assembly/3565.png
9/10/1963
English; Russian;
Political Force: We Continue the Change, 28.00 %; Profession: Lecturer;; 
range(0, 32)
**************************************************
Blagovest Kirilov Kirilov
Member
https://www.parliament.bg/images/Assembly/3783.png
12/10/1986
English; German; Russian;
Member of the previos NA: 45th NA, 46th NA,; Political Force: BSP for Bulgaria, 11.00 %; Profession: Enterpreneur; Engineer; Lecturer;; 
range(0, 26)
**************************************************
Bogomil Ivanov Petkov
Member
https://www.parliament.bg/images/Assembly/3793.png
4/10/1966
Russian;
Political Force: We Continue the Change, 28.0

range(0, 54)
**************************************************
Ekaterina Kirilova Dimitrusheva
Member
https://www.parliament.bg/images/Assembly/3542.png
13/11/1990
English; French; German; Russian;
Political Force: We Continue the Change, 28.00 %; Profession: Jurist;; 
range(0, 46)
**************************************************
Ekaterina Spasova Gecheva-Zaharieva
Member
https://www.parliament.bg/images/Assembly/3603.png
8/08/1975
English; German;
Member of the previos NA: 44th NA, 45th NA, 46th NA,; Political Force: GERB-UDF, 25.00 %; Profession: Jurist;; 
range(0, 28)
**************************************************
Elena Tsoneva Guncheva
Member
https://www.parliament.bg/images/Assembly/3619.png
7/01/1971
Political Force: Revival, 5.00 %; Profession: Jurist;; 
range(0, 18)
**************************************************
Elhan Mehmedov Kalkov
Member
https://www.parliament.bg/images/Assembly/3521.png
26/10/1973
Russian;
Member of the previos NA: 44th NA, 45th NA, 46th NA,; Pol

range(0, 18)
**************************************************
Hristo Bogomilov Terziyski
Member
https://www.parliament.bg/images/Assembly/3593.png
31/07/1968
English; Russian;
Member of the previos NA: 45th NA, 46th NA,; Political Force: GERB-UDF, 25.00 %; Profession: Engineer;; 
range(0, 18)
**************************************************
Hristo Georgiev Gadzhev
Member
https://www.parliament.bg/images/Assembly/3672.png
28/05/1983
English; French;
Member of the previos NA: 43rd NA, 44th NA, 45th NA, 46th NA,; Political Force: GERB-UDF, 25.00 %; 
range(0, 30)
**************************************************
Hristo Hristov Petrov
Member
https://www.parliament.bg/images/Assembly/3677.png
19/12/1979
English; Greek;
Political Force: We Continue the Change, 28.00 %; 
range(0, 10)
**************************************************
Hristo Lyubomirov Ivanov
Member
https://www.parliament.bg/images/Assembly/3622.png
13/09/1974
Member of the previos NA: 45th NA, 46th NA,; Political Force: D

Ivaylo Nikolaev Mirchev
Member
https://www.parliament.bg/images/Assembly/3547.png
6/05/1980
Member of the previos NA: 45th NA, 46th NA,; Political Force: DEMOCRATIC BULGARIA – UNION (Yes, Bulgaria!, Democrats for a Strong Bulgaria, Green Movement), 7.00 %; 
range(0, 30)
**************************************************
Ivaylo Valentinov Shotev
Member
https://www.parliament.bg/images/Assembly/3723.png
28/03/1987
English; Russian; Spanish;
Political Force: We Continue the Change, 28.00 %; Profession: Economist;; 
range(0, 24)
**************************************************
Ivelin Statev Ivanov
Member
https://www.parliament.bg/images/Assembly/3645.png
16/12/1970
Russian;
Political Force: GERB-UDF, 25.00 %; Profession: Engineer;; 
range(0, 14)
**************************************************
Ivo Georgiev Atanasov
Member
https://www.parliament.bg/images/Assembly/3576.png
29/05/1965
German;
Member of the previos NA: 45th NA, 46th NA,; Political Force: PP There Is Such People, 10.00 %; 

https://www.parliament.bg/images/Assembly/3614.png
29/03/1975
English; Russian;
Political Force: Movement for Rights and Freedoms - MRF, 14.00 %; Profession: Economist;; 
range(0, 74)
**************************************************
Mariya Shtereva Belova
Member
https://www.parliament.bg/images/Assembly/3649.png
25/04/1980
English; Greek;
Member of the previos NA: 43rd NA, 44th NA, 45th NA,; Political Force: GERB-UDF, 25.00 %; Profession: Jurist;; 
range(0, 18)
**************************************************
Martin Dimitrov Dimitrov
Member
https://www.parliament.bg/images/Assembly/3667.png
13/04/1977
Member of the previos NA: 40th NA, 41st NA, 43rd NA, 45th NA, 46th NA,; Political Force: DEMOCRATIC BULGARIA – UNION (Yes, Bulgaria!, Democrats for a Strong Bulgaria, Green Movement), 7.00 %; 
range(0, 8)
**************************************************
Maya Yordanova Dimitrova
Member
https://www.parliament.bg/images/Assembly/3586.png
9/09/1965
Russian;
Member of the previos NA: 45t

Petar Nikolaev Kulenski
Member
https://www.parliament.bg/images/Assembly/3606.png
24/01/1986
English; German;
Political Force: We Continue the Change, 28.00 %; Profession: Jurist; Economist;; 
range(0, 48)
**************************************************
Petar Nikolaev Nikolov
Member
https://www.parliament.bg/images/Assembly/3501.png
3/06/1979
Member of the previos NA: 46th NA,; Political Force: GERB-UDF, 25.00 %; 
range(0, 26)
**************************************************
Petar Pandushev Chobanov
Member
https://www.parliament.bg/images/Assembly/3683.png
20/07/1976
English; Russian;
Member of the previos NA: 42nd NA, 43rd NA, 46th NA,; Political Force: Movement for Rights and Freedoms - MRF, 14.00 %; Profession: Lecturer; Economist;; 
range(0, 52)
**************************************************
Petar Vasilev Kyosev
Member
https://www.parliament.bg/images/Assembly/3798.png
1/02/1989
English;
Political Force: We Continue the Change, 28.00 %; Profession: Jurist;; 
range(0, 38)
*

range(0, 16)
**************************************************
Stefan Ivanov Shilev
Member
https://www.parliament.bg/images/Assembly/3513.png
6/12/1973
English; Russian; Spanish;
Political Force: GERB-UDF, 25.00 %; Profession: Lecturer;; 
range(0, 30)
**************************************************
Stefan Nedelchev Mirev
Member
https://www.parliament.bg/images/Assembly/3604.png
6/09/1985
Member of the previos NA: 45th NA, 46th NA,; Political Force: GERB-UDF, 25.00 %; 
range(0, 26)
**************************************************
Stoil Miroslavov Stoilov
Member
https://www.parliament.bg/images/Assembly/3617.png
4/05/1992
English;
Political Force: We Continue the Change, 28.00 %; Profession: IT expert; Economist;; 
range(0, 12)
**************************************************
Stoyan Aleksandrov Mihalev
Member
https://www.parliament.bg/images/Assembly/3560.png
10/04/1972
Member of the previos NA: 45th NA, 46th NA,; Political Force: DEMOCRATIC BULGARIA – UNION (Yes, Bulgaria!, Demo

range(0, 38)
**************************************************
Vladislav Pantchev Panev
Member
https://www.parliament.bg/images/Assembly/3737.png
6/04/1976
Member of the previos NA: 45th NA, 46th NA,; Political Force: DEMOCRATIC BULGARIA – UNION (Yes, Bulgaria!, Democrats for a Strong Bulgaria, Green Movement), 7.00 %; 
range(0, 68)
**************************************************
Vyara Emilova Yordanova
Member
https://www.parliament.bg/images/Assembly/3595.png
9/06/1978
English;
Member of the previos NA: 45th NA,; Political Force: BSP for Bulgaria, 11.00 %; Profession: psychologist;; 
range(0, 26)
**************************************************
Yana Veselinova Balnikova
Member
https://www.parliament.bg/images/Assembly/3585.png
1/08/1980
English; French; Russian; Japanese;
Political Force: We Continue the Change, 28.00 %; Profession: Economist;; 
range(0, 2)
**************************************************
Yavor Rumenov Bozhankov
Member
https://www.parliament.bg/images/Assembly